#  MACHINE LEARNING MODEL

In [115]:
import pandas as pd
df = pd.read_csv('movies_data_eda.csv')

In [116]:
df = df[(df['budget'].isna()==False)&(df['revenue'].isna()==False)]

In [117]:
df.head()

,title,year,genres,director,original_language,budget,revenue,rating,age_rating,awarded,primary_genre,runtime_minutes,profit,roi,genre_count,budget_in_mil,revenue_in_mil,profit_in_mil,IsHit
4,Demon Slayer: Kimetsu no Yaiba Infinity Castle,2025,"Animation , Action , Fantasy",Haruo Sotozaki,Japanese,20000000.0,7.330302e+08,88,Adult,1,Animation,156,713030221.0,3565.151105,3,20.0,733.030221,713.030221,1
5,The Shawshank Redemption,1994,"Drama, Crime",Frank Darabont,English,25000000.0,2.834147e+07,87,Adult,1,Drama,142,3341469.0,13.365876,2,25.0,28.341469,3.341469,1
6,The Godfather,1972,"Drama, Crime",Francis Ford Coppola,English,6000000.0,2.450664e+08,87,Adult,1,Drama,175,239066411.0,3984.440183,2,6.0,245.066411,239.066411,1
7,Michael,2026,"Music, Drama",Antoine Fuqua,English,250000000.0,1.032616e+09,87,Teen,1,Music,128,782616139.0,313.046456,2,250.0,1032.616139,782.616139,1
8,Project Hail Mary,2026,"Science Fiction, Adventure",Christopher Miller,English,200000000.0,6.842347e+08,86,Teen,1,Science Fiction,157,484234722.0,242.117361,2,200.0,684.234722,484.234722,1


In [118]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1178 entries, 4 to 1998
Data columns (total 19 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   title              1178 non-null   object 
 1   year               1178 non-null   int64  
 2   genres             1178 non-null   object 
 3   director           1178 non-null   object 
 4   original_language  1178 non-null   object 
 5   budget             1178 non-null   float64
 6   revenue            1178 non-null   float64
 7   rating             1178 non-null   int64  
 8   age_rating         1178 non-null   object 
 9   awarded            1178 non-null   int64  
 10  primary_genre      1178 non-null   object 
 11  runtime_minutes    1178 non-null   int64  
 12  profit             1178 non-null   float64
 13  roi                1178 non-null   float64
 14  genre_count        1178 non-null   int64  
 15  budget_in_mil      1178 non-null   float64
 16  revenue_in_mil     1178 non-n

## Hit or Flop Movie Predictor

In [119]:
drop_cols = [
    'title',
    'director',
    'genres',
    'profit',
    'profit_in_mil',
    'roi',
    'revenue',
    'revenue_in_mil',
    'IsHit',
    'budget'
]

x = df.drop(columns=drop_cols)
y = df['IsHit']

In [120]:
x.columns

Index(['year', 'original_language', 'rating', 'age_rating', 'awarded',
       'primary_genre', 'runtime_minutes', 'genre_count', 'budget_in_mil'],
      dtype='object')

In [121]:
# Encoding the Categorical Columns
x = pd.get_dummies(x, columns=['primary_genre','original_language',
                               'age_rating'], drop_first=True, dtype=int)
x.head()

,year,rating,awarded,runtime_minutes,genre_count,budget_in_mil,primary_genre_Adventure,primary_genre_Animation,primary_genre_Comedy,primary_genre_Crime,...,original_language_Serbian,original_language_Spanish; Castilian,original_language_Swedish,original_language_Telugu,original_language_Thai,original_language_Turkish,original_language_Ukrainian,age_rating_Family Friendly,age_rating_Teen,age_rating_Unknown
4,2025,88,1,156,3,20.0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
5,1994,87,1,142,2,25.0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6,1972,87,1,175,2,6.0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
7,2026,87,1,128,2,250.0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
8,2026,86,1,157,2,200.0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0


In [122]:
# Training and Testing Split
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, y,
                                                    test_size=0.2,
                                                    random_state=42,
                                                    stratify=y
                                                   )

In [123]:
# Standardizing the Numerical cols in Testing and Training buckets
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

numeric_cols = ['year', 'rating', 'runtime_minutes', 'genre_count', 'budget_in_mil']

x_train[numeric_cols] = scaler.fit_transform(x_train[numeric_cols])
x_test[numeric_cols] = scaler.transform(x_test[numeric_cols])

## Logistic Regression Model

In [124]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

x_train_balanced, y_train_balanced = smote.fit_resample(
    x_train,
    y_train
)

In [125]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

model1 = LogisticRegression(random_state=42)

# Train
model1.fit(x_train_balanced, y_train_balanced)

# Predictions
y_pred = model1.predict(x_test)

# Metrics
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy : 0.690677966101695
Precision: 0.8809523809523809
Recall   : 0.736318407960199
F1 Score : 0.8021680216802168

Classification Report:
              precision    recall  f1-score   support

           0       0.22      0.43      0.29        35
           1       0.88      0.74      0.80       201

    accuracy                           0.69       236
   macro avg       0.55      0.58      0.55       236
weighted avg       0.78      0.69      0.73       236


Confusion Matrix:
[[ 15  20]
 [ 53 148]]


In [126]:
# Insights

# After balancing the training data, Logistic Regression's
# accuracy decreased from 85.2% to 69.1%, but its ability to
# identify Flop movies improved substantially,
# with Flop recall increasing from 6% to 42.9%.

## Decision Tree Classifier Model

In [127]:
from sklearn.tree import DecisionTreeClassifier

model2 = DecisionTreeClassifier(
    random_state=42
)

# Train on balanced training data
model2.fit(x_train_balanced, y_train_balanced)

# Predictions
y_pred_dt = model2.predict(x_test)

# Metrics
print("Accuracy :", accuracy_score(y_test, y_pred_dt))
print("Precision:", precision_score(y_test, y_pred_dt))
print("Recall   :", recall_score(y_test, y_pred_dt))
print("F1 Score :", f1_score(y_test, y_pred_dt))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_dt))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_dt))

Accuracy : 0.7330508474576272
Precision: 0.8833333333333333
Recall   : 0.7910447761194029
F1 Score : 0.8346456692913385

Classification Report:
              precision    recall  f1-score   support

           0       0.25      0.40      0.31        35
           1       0.88      0.79      0.83       201

    accuracy                           0.73       236
   macro avg       0.57      0.60      0.57       236
weighted avg       0.79      0.73      0.76       236


Confusion Matrix:
[[ 14  21]
 [ 42 159]]


In [128]:
# Insights

# Decision Tree achieved 73.3% accuracy and an F1-score of 83.5%
# for Hit movies, outperforming the balanced Logistic Regression model.
# However, its ability to identify Flop movies remained limited,
# with a recall of 40%

## Random Forest Classifier Model

In [133]:
from sklearn.ensemble import RandomForestClassifier

model3 = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

# Train on balanced training data
model3.fit(x_train_balanced, y_train_balanced)

# Predictions
y_pred_rf = model3.predict(x_test)

# Metrics
print("Accuracy :", accuracy_score(y_test, y_pred_rf))
print("Precision:", precision_score(y_test, y_pred_rf))
print("Recall   :", recall_score(y_test, y_pred_rf))
print("F1 Score :", f1_score(y_test, y_pred_rf))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

Accuracy : 0.7627118644067796
Precision: 0.864321608040201
Recall   : 0.8557213930348259
F1 Score : 0.86

Classification Report:
              precision    recall  f1-score   support

           0       0.22      0.23      0.22        35
           1       0.86      0.86      0.86       201

    accuracy                           0.76       236
   macro avg       0.54      0.54      0.54       236
weighted avg       0.77      0.76      0.77       236


Confusion Matrix:
[[  8  27]
 [ 29 172]]


In [134]:
# Insights

# Random Forest achieved the highest overall accuracy (76.3%)
# and Hit-class F1-score (86.0%) among the three models tested.
# However, its ability to identify Flop movies was limited, with only 22.9% recall.

In [139]:
# If we see the Avg F1 score then Decision Tree Classifier is the most balanced model out
# of these 3 models with 73.3 % Accuracy and flop movie identification 40% Recall